<a href="https://colab.research.google.com/github/haris444/autonomous-agents/blob/dev/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAC Ablation Study — Round 2

Building on Round 1 results (A/B/C/D), which showed:
- Social rewards barely matter (D = A)
- Predator hurts performance (C >> A)
- Lifesteal irrelevant (B = A)
- **Untested: does the LEDGER (social memory) matter?**

Round 2 experiments:
- **B: No Ledger** — zero social channels in observations. Tests the project's core contribution.
- **E: No Rich Food** — remove cooperative food. Tests if environmental pressure drives cooperation.
- **F: No Ledger + No Social** — double ablation. Pure survival, no social anything.
- **G: 3 Predators** — heavy external threat. Does MORE pressure force cooperation?

**Runtime:** ~2-3 hours on T4 GPU (4 runs x 5M steps each, 4 envs per run)

## 1. Setup

In [ ]:
# Clone repo and install deps
!git clone -b dev https://github.com/haris444/autonomous-agents.git
%cd autonomous-agents
!pip install -q pyyaml

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Base Config + A/B Variants

In [ ]:
#@title Sweep Settings {run: "auto"}
TOTAL_TIMESTEPS = 5_000_000   #@param {type: "integer"}
N_ENVS = 4                    #@param [2, 4, 8] {type: "raw"}
CKPT_EVERY = 500              #@param {type: "integer"}

SWEEP_DIR = "results/ab_test_r2"

In [ ]:
import yaml, os, copy

os.makedirs(SWEEP_DIR, exist_ok=True)

# ── Base config (matches sac_phase11_lifesteal.yaml) ──
BASE = {
    'experiment': {'name': 'ab_r2', 'description': 'Ablation study round 2'},
    # Environment
    'grid_size': 8,
    'n_agents': 8,
    'max_hp': 100.0,
    'hp_decay_rate': 0.01,
    'attack_damage_fraction': 0.125,
    'lifesteal_fraction': 0.25,
    'max_steps_per_episode': 128,
    # Entity tokens
    'max_food_tokens': 16,
    'fourier_bands': 4,
    'attention_embed_dim': 64,
    'attention_num_heads': 4,
    'group_embed_dim': 16,
    # Food
    'poor_food_value': 10.0,
    'rich_food_value': 30.0,
    'poor_food_spawn_rate': 0.000336,
    'rich_food_spawn_rate': 0.0084,
    'food_coverage_cap': 0.40,
    # Rewards
    'r_small': 10.0,
    'r_large': 20.0,
    'r_attack_mult': 0.1,
    'r_damage_taken': -1.0,
    'r_low_hp': 0.0,
    'r_food_share': 0.0,
    'r_betrayal': 0.0,
    'r_reciprocity': 0.0,
    'r_defense': 0.0,
    'r_revenge': 0.5,
    'r_survival': 0.0,
    'r_death': 0.0,
    'r_coop_attempt': 2.0,
    'r_approach_food': 0.1,
    'r_ally_proximity': 0.0,
    # Social hierarchy
    'r_hierarchy': 0.25,
    'hierarchy_food_weight': 1.0,
    'hierarchy_damage_weight': 1.0,
    'hierarchy_hp_weight': 1.0,
    'hierarchy_kill_weight': 2.0,
    # SAC hyperparameters
    'sac_learning_rate': 0.001,
    'sac_tau': 0.01,
    'sac_alpha_init': 0.8,
    'sac_auto_alpha': True,
    'sac_target_entropy_scale': 0.1,
    'sac_gamma': 0.95,
    'sac_buffer_size': 500_000,
    'sac_batch_size': 2048,
    'sac_learning_starts': 50_000,
    'sac_update_frequency': 4,
    'sac_utd_ratio': 1,
    'sac_target_update_interval': 1,
    # Training
    'total_timesteps': TOTAL_TIMESTEPS,
    'seed': 42,
    'n_envs': N_ENVS,
    # Predator
    'n_predators': 1,
    'predator_hp_mult': 0.5,
    'predator_damage': 10.0,
    'predator_kill_reward': 5.0,
    'predator_respawn_steps': 20,
    # No curriculum
    'pretrain_mode': False,
    'curriculum_enabled': False,
}

# ── 4 Round 2 variants ──
VARIANTS = {
    # B: Zero social channels in observations — agents can't see interaction history
    'B_no_ledger': {
        'ablate_ledger': True,
    },
    # E: Remove rich food — no environmental pressure to cooperate
    'E_no_rich_food': {
        'rich_food_spawn_rate': 0.0,
        'r_large': 0.0,
    },
    # F: No ledger AND no social rewards — pure survival, no social anything
    'F_no_ledger_no_social': {
        'ablate_ledger': True,
        'r_hierarchy': 0.0,
        'r_revenge': 0.0,
        'r_coop_attempt': 0.0,
        'r_reciprocity': 0.0,
        'r_defense': 0.0,
        'r_betrayal': 0.0,
        'r_food_share': 0.0,
        'r_ally_proximity': 0.0,
    },
    # G: 3 predators with reduced damage — heavy external threat but survivable
    'G_3_predators': {
        'n_predators': 3,
        'predator_damage': 5.0,
    },
}

# ── Generate config YAMLs ──
sweep_configs = {}
for name, overrides in VARIANTS.items():
    variant = copy.deepcopy(BASE)
    variant['experiment']['name'] = name
    for k, v in overrides.items():
        variant[k] = v

    out_dir = f'{SWEEP_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)
    variant['training'] = {'output_dir': out_dir}

    cfg_path = f'{out_dir}/config.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(variant, f, default_flow_style=False)
    sweep_configs[name] = {'config': cfg_path, 'csv': f'{out_dir}/log.csv', 'dir': out_dir}

print(f'Round 2 Ablation: {len(VARIANTS)} variants x {TOTAL_TIMESTEPS:,} steps x {N_ENVS} envs')
print()
for name, paths in sweep_configs.items():
    diff = VARIANTS[name]
    desc = ', '.join(f'{k}={v}' for k, v in diff.items())
    print(f'  {name}: {desc}')

## 3. Run All 4 Experiments in Parallel

In [ ]:
import subprocess, time

processes = {}
for name, paths in sweep_configs.items():
    cmd = [
        'python', '-m', 'training.train_sac',
        '--experiment', paths['config'],
        '--csv-log', paths['csv'],
        '--ckpt-every', str(CKPT_EVERY),
    ]
    log_file = open(f'{paths["dir"]}/stdout.log', 'w')
    p = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
    processes[name] = {'proc': p, 'log': log_file}
    print(f'Launched {name} (PID {p.pid})')

print(f'\n{len(processes)} runs launched. Monitoring...\n')

while True:
    time.sleep(60)
    alive = {n: p for n, p in processes.items() if p['proc'].poll() is None}
    done = len(processes) - len(alive)
    status = f'[{time.strftime("%H:%M:%S")}] {done}/{len(processes)} done'

    for name in list(sweep_configs.keys()):
        csv_path = sweep_configs[name]['csv']
        try:
            with open(csv_path) as f:
                lines = f.readlines()
                if len(lines) > 1:
                    last = lines[-1].strip().split(',')
                    ret, step, ep = last[2], last[1], last[3]
                    status += f' | {name}: R={ret} ep={ep}'
        except:
            pass
    print(status)

    if not alive:
        break

for p in processes.values():
    p['log'].close()

print('\nAll runs complete!')
for name, p in processes.items():
    rc = p['proc'].returncode
    print(f'  {name}: {"OK" if rc == 0 else f"FAILED (rc={rc})"}')

## 4. Comparison Plots

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

colors = {
    'B_no_ledger': '#ff7f0e',
    'E_no_rich_food': '#2ca02c',
    'F_no_ledger_no_social': '#d62728',
    'G_3_predators': '#9467bd',
}
labels = {
    'B_no_ledger': 'B: No Ledger',
    'E_no_rich_food': 'E: No Rich Food',
    'F_no_ledger_no_social': 'F: No Ledger + No Social',
    'G_3_predators': 'G: 3 Predators',
}

dfs = {}
for name, paths in sweep_configs.items():
    try:
        dfs[name] = pd.read_csv(paths['csv'])
    except Exception as e:
        print(f'Warning: {name} -- {e}')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Ablation Study Round 2', fontsize=14, fontweight='bold')

# 1. Episode Return
ax = axes[0, 0]
for name, df in dfs.items():
    ax.plot(df['global_step'], df['episode_return'].rolling(50).mean(),
            color=colors[name], linewidth=2, label=labels[name])
ax.set_xlabel('Global Step')
ax.set_ylabel('Episode Return')
ax.set_title('Episode Return (smoothed)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Q-values
ax = axes[0, 1]
for name, df in dfs.items():
    q1 = df['q1_mean'].replace(0, float('nan'))
    ax.plot(df['global_step'], q1.rolling(50).mean(),
            color=colors[name], linewidth=2, label=labels[name])
ax.set_xlabel('Global Step')
ax.set_ylabel('Q1 Mean')
ax.set_title('Critic Q-Values')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Alpha (direction)
ax = axes[1, 0]
for name, df in dfs.items():
    ad = df['alpha_dir'].replace(0, float('nan'))
    ax.plot(df['global_step'], ad, color=colors[name], linewidth=1.5, label=labels[name])
ax.set_xlabel('Global Step')
ax.set_ylabel('Alpha (direction)')
ax.set_title('Entropy Temperature (Direction)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 4. Alpha (action type)
ax = axes[1, 1]
for name, df in dfs.items():
    aa = df['alpha_act'].replace(0, float('nan'))
    ax.plot(df['global_step'], aa, color=colors[name], linewidth=1.5, label=labels[name])
ax.set_xlabel('Global Step')
ax.set_ylabel('Alpha (action type)')
ax.set_title('Entropy Temperature (Action Type)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SWEEP_DIR}/ab_r2_comparison.png', dpi=150)
plt.show()

# Summary table
print('\nRound 2 Ablation Results:')
print(f'{"Variant":<30} {"Final Return":>14} {"Episodes":>10} {"Final Q1":>10}')
print('-' * 66)
for name, df in dfs.items():
    ret = df['episode_return'].iloc[-1]
    eps = df['episodes_completed'].iloc[-1]
    q1 = df['q1_mean'].iloc[-1]
    print(f'{labels[name]:<30} {ret:>14.1f} {eps:>10} {q1:>10.2f}')

## 5. Save to Google Drive

In [ ]:
from google.colab import drive
import shutil, glob

drive.mount('/content/drive')

drive_dir = '/content/drive/MyDrive/AutonomousAgents/ab_test_r2'
os.makedirs(drive_dir, exist_ok=True)

# Copy comparison plot
plot_path = f'{SWEEP_DIR}/ab_r2_comparison.png'
if os.path.exists(plot_path):
    shutil.copy2(plot_path, drive_dir)

# Copy everything useful for each variant
for name, paths in sweep_configs.items():
    var_drive = f'{drive_dir}/{name}'
    os.makedirs(var_drive, exist_ok=True)

    for f in ['config.yaml', 'log.csv', 'stdout.log', 'training_curves.png']:
        p = f'{paths["dir"]}/{f}'
        if os.path.exists(p):
            shutil.copy2(p, var_drive)

    ckpts = sorted(glob.glob(f'{paths["dir"]}/checkpoint_*.pt'))
    for ckpt in ckpts:
        shutil.copy2(ckpt, var_drive)
    n_ckpts = len(ckpts)
    first = os.path.basename(ckpts[0]) if ckpts else 'none'
    last = os.path.basename(ckpts[-1]) if ckpts else 'none'
    print(f'{name}: {n_ckpts} checkpoints ({first} -> {last})')

print(f'\nAll results saved to: {drive_dir}')
for name in sweep_configs:
    var_drive = f'{drive_dir}/{name}'
    files = os.listdir(var_drive) if os.path.exists(var_drive) else []
    print(f'  {name}: {len(files)} files')